In [41]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # type: ignore
from tensorflow.keras.models import Sequential  # type: ignore
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization # type: ignore
from tensorflow.keras.optimizers import Adam # type: ignore
from tensorflow.keras.models import load_model # type: ignore
import cv2
import numpy as np

In [ ]:
data_dir = "/Users/raghava/Documents/Projects/4-Report Generator and Load Runner/charts_classification_training_images"
model_name = "chart_classification_model.h5"
batch_size = 8
img_height, img_width = 256,128

In [43]:
# Image size close to original aspect ratio (2:1 ratio like 1783x883)

datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    validation_split=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
)

train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

model = Sequential([
    # First Conv Block
    Conv2D(32, (3, 3), activation='relu', input_shape=(256, 128, 3)),  # input shape: (height, width, channels)
    MaxPooling2D(pool_size=(2, 2)),

    # Second Conv Block
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Third Conv Block
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Flatten and Fully Connected Layers
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')  # change 3 to the number of classes in your case
])


# Compile the model with a lower learning rate for better convergence
model.compile(optimizer=Adam(learning_rate=0.0005), loss='categorical_crossentropy', metrics=['accuracy'])


Found 720 images belonging to 3 classes.
Found 180 images belonging to 3 classes.


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [44]:
print(train_generator.class_indices)

# Invert the dictionary to get index-to-name mapping
index_to_class = {v: k for k, v in train_generator.class_indices.items()}

# Optional: print as a list
classes = [index_to_class[i] for i in range(len(index_to_class))]
print(classes)


{'leak': 0, 'no_issue': 1, 'spikes': 2}
['leak', 'no_issue', 'spikes']


In [45]:
# Train the model
model.fit(train_generator, validation_data=val_generator, epochs=10)

Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


90/90 ━━━━━━━━━━━━━━━━━━━━ 14s 120ms/step - accuracy: 0.4340 - loss: 1.1314 - val_accuracy: 0.7833 - val_loss: 0.5341
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 18s 110ms/step - accuracy: 0.7887 - loss: 0.5231 - val_accuracy: 0.8444 - val_loss: 0.3773
Epoch 3/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 10s 113ms/step - accuracy: 0.8445 - loss: 0.4535 - val_accuracy: 0.8444 - val_loss: 0.4284
Epoch 4/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 10s 111ms/step - accuracy: 0.8774 - loss: 0.3408 - val_accuracy: 0.9389 - val_loss: 0.2305
Epoch 5/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 10s 110ms/step - accuracy: 0.8985 - loss: 0.3278 - val_accuracy: 0.9167 - val_loss: 0.2158
Epoch 6/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 9s 100ms/step - accuracy: 0.8793 - loss: 0.3024 - val_accuracy: 0.9222 - val_loss: 0.2081
Epoch 7/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 10s 109ms/step - accuracy: 0.8984 - loss: 0.2278 - val_accuracy: 0.9500 - val_loss: 0.1691
Epoch 8/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 10s 109ms/step - accuracy: 0.9408 - loss: 0.1738 - val_accuracy: 0.9722

In [46]:
# Save the model
model.save(model_name)

In [47]:
# Set your image dimensions here (must match model input)

def predict(image_path, loaded_model):
    def preprocess_image(image_path):
        img = cv2.imread(image_path)  # Load the image
        # if img is None:
        #     return f"cv2 failed to read the image. Path exists? {os.path.exists(image_path)}. path : {image_path} Exact path: {repr(image_path)}"
        # else:
            # return f"Image loaded successfully. Shape: {img.shape}"
        img = cv2.resize(img, (img_width, img_height))  # Resize to (width, height)
        img = img / 255.0  # Normalize pixel values
        img = np.expand_dims(img, axis=0)  # Add batch dimension
        return img

    processed_image = preprocess_image(image_path)

    prediction = loaded_model.predict(processed_image)
    print(prediction)
    class_index = np.argmax(prediction)

    predicted_label = classes[class_index]

    return_string = f"Predicted Class: {predicted_label}.\nConfidence {prediction[0][class_index]*100:.2f}%\n"
    # for i, label in enumerate(classes):
    #     return_string += f"{label}: {prediction[0][i]*100:.2f}% confidence\n"

    return return_string


In [50]:
loaded_model = load_model(model_name)
print(predict("/content/charts_classification_training_images/leak/leak_100.png", loaded_model))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 311ms/step
[[1.000000e+00 6.290833e-20 4.130827e-12]]
Predicted Class: leak.
Confidence 100.00%



In [37]:
# import shutil
# shutil.rmtree("charts_classification_training_images")